# Chapter 4: Training a Neural Network and Computational Thinking

In [43]:
import torch
import matplotlib.pyplot as plt
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
# At the top of your notebook, add:
np.set_printoptions(suppress=True, precision=8)
torch.set_printoptions(sci_mode=False, precision=8)

cuda


In [11]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

In [12]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

In [13]:
df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE486586_FA25/refs/heads/main/Data/Hydropower.csv")
df

FacilityName,BCR,AnnualProduction,ConstructionCost,DesignHead,Y,X1,X2,X3
str,f64,i64,f64,i64,f64,f64,f64,f64
"""Anchor Dam """,0.02,126,5656.5,60,-3.912023,4.836282,8.640561,4.094345
"""Angostura Dam """,0.9,3218,3179.2,119,-0.105361,8.076515,8.064385,4.779123
"""Barretts Diversion Dam """,0.35,546,1391.4,15,-1.049822,6.302619,7.238066,2.70805
"""Belle Fourche Dam """,0.49,1319,2376.3,50,-0.71335,7.184629,7.7733,3.912023
"""Bonny Dam """,0.15,238,1476.8,70,-1.89712,5.472271,7.297633,4.248495
…,…,…,…,…,…,…,…,…
"""Upper Diamond Fork Flow Contro…",2.36,52161,22058.5,547,0.858662,10.86209,10.001453,6.304449
"""Upper Stillwater Dam """,0.32,1904,6064.5,161,-1.139434,7.551712,8.710207,5.081404
"""Vega Dam """,0.51,1702,3012.5,90,-0.673345,7.439559,8.010526,4.49981


we want to estimat annuam production based BCR, ConstructionCost and DesignHead

In [63]:
class SimpleNN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNN, self).__init__()
        self.in_features = in_features
        self.fc1 = nn.Linear(self.in_features, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [64]:
# Convert Polars DataFrame to numpy arrays
X = df.drop(['FacilityName', 'Y', 'X1', 'X2', 'X3', 'AnnualProduction']).to_numpy() 
y = df['AnnualProduction'].to_numpy()     

In [65]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [67]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32, device=device)

In [68]:
X_train_tensor

tensor([[-0.42984405, -0.68700987, -0.14693090],
        [-0.80440408,  2.90502715,  0.32870990],
        [ 3.12033367,  5.82382965,  0.22060971],
        [-0.86954492,  0.01938729,  0.31789988],
        [-0.78811884, -1.04316974, -0.73067188],
        [-0.91840059,  0.54276216,  0.05845944],
        [-0.73926318, -0.42819089, -0.80634201],
        [-0.85325974, -0.66431010, -0.70905185],
        [ 0.20527947,  0.85989898, -0.14693090],
        [-0.16928056, -0.49544418,  0.38275999],
        [ 0.46584296, -0.25573429,  0.37194997],
        [-0.78811884, -0.70036149, -0.36313128],
        [ 1.93151259,  0.37715679,  0.92326093],
        [-1.01611197, -1.07379389, -0.56852162],
        [ 0.87297344, -0.19735454,  0.59896034],
        [ 0.46584296,  2.01117086, -0.26584110],
        [ 0.88925862, -0.25589940, -0.36313128],
        [ 0.92182904,  0.53275359, -0.71986187],
        [ 0.40070209, -0.37059528,  0.56653029],
        [ 1.11725175,  0.02793067,  0.65301043],
        [-1.04868233

In [69]:
model = SimpleNN(in_features = X_train_tensor.shape[1]).to(device)


In [70]:
# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)
# Train the model
epochs = 100000

losses = torch.zeros(epochs, device=device)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    
    losses[epoch] = loss
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
        print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.5f} MB')

/home/refulgent/VersionControl/CPE487587_SP26/Code/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([152])) that is different to the input size (torch.Size([152, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1000/100000, Loss: 159494416.0
GPU Memory: 18.39502 MB
Epoch 2000/100000, Loss: 159482784.0
GPU Memory: 18.39502 MB
Epoch 3000/100000, Loss: 159459552.0
GPU Memory: 18.39502 MB
Epoch 4000/100000, Loss: 159422000.0
GPU Memory: 18.39502 MB
Epoch 5000/100000, Loss: 159367520.0
GPU Memory: 18.39502 MB
Epoch 6000/100000, Loss: 159293744.0
GPU Memory: 18.39502 MB
Epoch 7000/100000, Loss: 159197952.0
GPU Memory: 18.39502 MB
Epoch 8000/100000, Loss: 159077216.0
GPU Memory: 18.39502 MB
Epoch 9000/100000, Loss: 158928672.0
GPU Memory: 18.39502 MB
Epoch 10000/100000, Loss: 158749360.0
GPU Memory: 18.39502 MB
Epoch 11000/100000, Loss: 158536032.0
GPU Memory: 18.39502 MB
Epoch 12000/100000, Loss: 158285728.0
GPU Memory: 18.39502 MB
Epoch 13000/100000, Loss: 157995520.0
GPU Memory: 18.39502 MB
Epoch 14000/100000, Loss: 157662400.0
GPU Memory: 18.39502 MB
Epoch 15000/100000, Loss: 157283632.0
GPU Memory: 18.39502 MB
Epoch 16000/100000, Loss: 156856544.0
GPU Memory: 18.39502 MB
Epoch 17000/10000